# 01 - Data check

CholecSeg8k: 8,080 frames from 17 laparoscopic cholecystectomy videos (Cholec80), 13 classes. This notebook

1. lists every frame with its video and clip,
2. checks how the class labels are stored in the masks,
3. counts the pixels of each class,
4. fixes the split by video with the rule in `docs/plan.md`, and defines the small and the large classes for H1.

It looks only at the masks of the training and validation videos when it counts pixels for H1. Nothing is trained here.

Kaggle: no accelerator needed. **+ Add Input > Datasets > `newslab/cholecseg8k`** (the script `scripts/run_on_kaggle.py` does this by itself).
Output: `out/frames.csv`, `out/frames_per_video.csv`, `out/mapping_check.csv`, `out/mapping.json`, `out/class_pixels.csv`, `out/classes.json`, `out/split.csv`, `out/examples.png`, zipped as `results_01.zip`.

In [ ]:
import glob, json, math, os, re, shutil, time
from pathlib import Path
from multiprocessing import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

KAGGLE = Path('/kaggle').exists()
WORK = Path('/kaggle/working') if KAGGLE else Path('.')
DATA = Path(os.environ.get('DATA_ROOT', '/kaggle/input'))     # where the CholecSeg8k frames are
INPUT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))   # where the output of the previous notebook is
OUT = WORK / 'out'
OUT.mkdir(exist_ok=True)
H, W = 256, 448   # frames are 480 x 854; resized to a size the network can halve five times

CLASSES = ['black background', 'abdominal wall', 'liver', 'gastrointestinal tract', 'fat', 'grasper',
           'connective tissue', 'blood', 'cystic duct', 'L-hook electrocautery', 'gallbladder',
           'hepatic vein', 'liver ligament']
N_CLASSES = len(CLASSES)
IGNORE = 255      # mask pixels with a value that belongs to no class


def find_input(name):
    """Find a file from the previous notebook's output under INPUT."""
    hits = sorted(glob.glob(str(INPUT / '**' / name), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {INPUT}: add the output of the previous notebook as input')
    return Path(hits[0])

## 1. Frames

Every frame has four files: the image (`..._endo.png`), a colour mask for viewing, the mask from the annotation tool, and the "watershed" mask, which the authors made for programs. The video is the folder named `videoNN`; a clip is 80 frames in a row.

In [ ]:
paths = sorted(glob.glob(str(DATA / '**' / '*_endo.png'), recursive=True))
rows = []
for p in paths:
    video = next((x for x in Path(p).parts if re.fullmatch(r'video\d+', x)), None)
    m = re.search(r'frame_(\d+)_endo\.png$', Path(p).name)
    rows.append(dict(path=p, video=video, clip_name=Path(p).parent.name, frame=int(m.group(1)) if m else -1))
frames = pd.DataFrame(rows)
assert len(frames), f'no *_endo.png files under {DATA}: is the dataset added as input?'
assert frames.video.notna().all(), 'some frames are not inside a videoNN folder'
before = len(frames)
frames = frames.drop_duplicates(['video', 'clip_name', 'frame']).reset_index(drop=True)   # in case the dataset is mounted twice
missing = [p for p in frames.path if not Path(p.replace('_endo.png', '_endo_watershed_mask.png')).exists()]
print(len(frames), 'frames', f'({before - len(frames)} duplicates dropped),', frames.video.nunique(), 'videos,',
      frames.clip_name.nunique(), 'clips,', len(missing), 'without watershed mask')
assert not missing, missing[:5]

per_video = frames.groupby('video').agg(clips=('clip_name', 'nunique'), frames=('path', 'size')).reset_index()
per_video.to_csv(OUT / 'frames_per_video.csv', index=False)
print(per_video.to_string(index=False))

## 2. How are the classes stored?

The paper says that in the watershed mask every pixel holds its class ID in all three channels. Code written by others for this dataset uses other values (for example 50 for the black background and 11 for the abdominal wall). So the notebook looks at the values that are really there, on 300 random frames:

- Do the three channels always hold the same value?
- Which values occur, and how often?
- Which colour of the colour mask sits on each value? This links a value to a class name through the colours in the paper's figure.

If all frequent values are in the expected list, that list is used. If they are 0-12, the values are the class IDs themselves. Values that the colour mask draws in white belong to no class; they are left out of training and scoring. Anything else stops the notebook, so that no model is trained on wrong labels.

In [ ]:
EXPECTED = {50: 0, 11: 1, 21: 2, 13: 3, 12: 4, 31: 5, 23: 6, 24: 7, 25: 8, 32: 9, 22: 10, 33: 11, 5: 12}
EXPECTED_COLOUR = {0: '#7f7f7f', 1: '#d28c8c', 2: '#ff7272', 3: '#e7469c', 4: '#bab74b', 5: '#aaff00', 6: '#ff5500',
                   7: '#ff0000', 8: '#ffff00', 9: '#a9ffb8', 10: '#ffa0a5', 11: '#003280', 12: '#6f4a00'}

rng = np.random.default_rng(0)
sample = frames.path.values[rng.choice(len(frames), size=min(300, len(frames)), replace=False)]
value_px, colour_px, channel_mismatch = {}, {}, 0
for p in sample:
    ws = np.array(Image.open(p.replace('_endo.png', '_endo_watershed_mask.png')))
    if ws.ndim == 3:
        channel_mismatch += int((ws[..., :3] != ws[..., :1]).any(-1).sum())
        ws = ws[..., 0]
    col = np.array(Image.open(p.replace('_endo.png', '_endo_color_mask.png')).convert('RGB'))
    if col.shape[:2] != ws.shape:
        col = np.array(Image.fromarray(col).resize(ws.shape[::-1], Image.NEAREST))
    for v in np.unique(ws):
        sel = ws == v
        value_px[int(v)] = value_px.get(int(v), 0) + int(sel.sum())
        c, n = np.unique(col[sel].reshape(-1, 3), axis=0, return_counts=True)
        for rgb, k in zip(c, n):
            key = (int(v), '#%02x%02x%02x' % tuple(int(a) for a in rgb))
            colour_px[key] = colour_px.get(key, 0) + int(k)

total = sum(value_px.values())
check = []
for v, n in sorted(value_px.items(), key=lambda kv: -kv[1]):
    cols = {c: k for (vv, c), k in colour_px.items() if vv == v}
    top = max(cols, key=cols.get)
    cls = EXPECTED.get(v)
    check.append(dict(value=v, share=n / total, top_colour=top, top_colour_share=cols[top] / n,
                      expected_class=None if cls is None else CLASSES[cls],
                      expected_colour=None if cls is None else EXPECTED_COLOUR[cls]))
check = pd.DataFrame(check)
check.to_csv(OUT / 'mapping_check.csv', index=False)
print('pixels where the three channels differ:', channel_mismatch)
print(check.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
# Pixels drawn white in the colour mask carry no class (not labelled). They are left out, as the plan says.
unlabelled = set(check.loc[check.top_colour == '#ffffff', 'value'])
frequent = set(check.loc[check.share >= 0.001, 'value']) - unlabelled
if frequent <= set(EXPECTED):
    mapping, rule = EXPECTED, 'values as in EXPECTED'
elif frequent <= set(range(N_CLASSES)):
    mapping, rule = {v: v for v in range(N_CLASSES)}, 'values are the class IDs (0-12)'
else:
    raise ValueError(f'unexpected mask values {sorted(frequent - set(EXPECTED))}; see mapping_check.csv')
mapping = {v: c for v, c in mapping.items() if v not in unlabelled}
print('values shown white in the colour mask (no class, left out):', sorted(unlabelled))
rare = check[~check.value.isin(list(mapping))]
colour_ok = (check.dropna(subset=['expected_colour'])
             .assign(ok=lambda d: d.top_colour == d.expected_colour)).ok.mean() if rule.startswith('values as') else np.nan
print('rule:', rule)
print(f'share of pixels with a value outside the mapping (ignored in training and scoring): {rare.share.sum():.5f}')
print(f'values whose most common colour is the expected one: {colour_ok:.0%}')
_ = (OUT / 'mapping.json').write_text(json.dumps({str(k): v for k, v in mapping.items()}, indent=1))

## 3. Split by video

The rule from `docs/plan.md`, fixed before this notebook was run:

- The video names are sorted and shuffled with `numpy.random.default_rng(20261114)`. The first 3 are the **test videos**, the next 3 the **validation videos** of model B, the other 11 its training videos.
- **Model A** uses the same 14 non-test videos, but its validation frames are 15 % of all their frames, drawn at random with `default_rng(20261115)`.

The test videos are never loaded in notebooks 02 and 03.

In [ ]:
videos = np.array(sorted(frames.video.unique()))
order = np.random.default_rng(20261114).permutation(videos)
test_v, val_v = sorted(str(v) for v in order[:3]), sorted(str(v) for v in order[3:6])
frames['split_B'] = np.where(frames.video.isin(test_v), 'test', np.where(frames.video.isin(val_v), 'val', 'train'))
non_test = np.flatnonzero(frames.split_B.values != 'test')
pick = np.random.default_rng(20261115).permutation(non_test)[:round(0.15 * len(non_test))]
frames['split_A'] = np.where(frames.split_B == 'test', 'test', 'train')
frames.loc[pick, 'split_A'] = 'val'
frames.to_csv(OUT / 'split.csv', index=False)
print('test videos:', test_v)
print('validation videos (model B):', val_v)
print(pd.crosstab(frames.split_A, frames.split_B, margins=True))

## 4. Pixels per class

Counted on all frames of the 14 non-test videos, after resizing to 256 x 448 as in training. For H1, a class is **large** if it covers at least 5 % of these pixels and **small** if it covers less than 1 %. The black background (class 0) is the dark border of the endoscope picture and is left out.

In [ ]:
LUT = np.full(256, IGNORE, np.uint8)
for value, cls in mapping.items():
    LUT[value] = cls


def class_counts(path):
    m = np.array(Image.open(path.replace('_endo.png', '_endo_watershed_mask.png')))
    if m.ndim == 3:
        m = m[..., 0]
    m = np.asarray(Image.fromarray(LUT[m]).resize((W, H), Image.NEAREST))
    return np.bincount(m.ravel(), minlength=256)[:N_CLASSES]


with Pool(os.cpu_count()) as pool:
    counts = np.stack(pool.map(class_counts, frames.path.tolist(), chunksize=16))
frames_counts = pd.DataFrame(counts, columns=CLASSES)
use = frames.split_B.values != 'test'
px = frames_counts[use].sum()
share = px.iloc[1:] / px.iloc[1:].sum()
in_frames = (frames_counts[use] > 0).mean()
table = pd.DataFrame({'class_id': range(N_CLASSES), 'pixels': px.values,
                      'share_without_border': [np.nan] + list(share.values),
                      'frames_with_class': in_frames.values})
table.to_csv(OUT / 'class_pixels.csv', index=False)
print(table.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

large = [i for i in range(1, N_CLASSES) if share.iloc[i - 1] >= 0.05]
small = [i for i in range(1, N_CLASSES) if share.iloc[i - 1] < 0.01]
_ = (OUT / 'classes.json').write_text(json.dumps({'large': large, 'small': small,
                                              'names': CLASSES, 'test_videos': test_v, 'val_videos_B': val_v}, indent=1))
print('large:', [CLASSES[i] for i in large])
print('small:', [CLASSES[i] for i in small])

## 5. A look at the data

Four frames from different videos with their class masks.

In [ ]:
cmap = plt.get_cmap('tab20', N_CLASSES)
ex = frames[frames.split_B != 'test'].groupby('video').head(1).sample(4, random_state=0)
fig, ax = plt.subplots(2, 4, figsize=(16, 5))
for j, p in enumerate(ex.path):
    img = Image.open(p).convert('RGB').resize((W, H))
    m = np.array(Image.open(p.replace('_endo.png', '_endo_watershed_mask.png')))
    m = m[..., 0] if m.ndim == 3 else m
    m = np.asarray(Image.fromarray(LUT[m]).resize((W, H), Image.NEAREST)).astype(float)
    m[m == IGNORE] = np.nan
    ax[0, j].imshow(img); ax[0, j].set_title(Path(p).parent.name, fontsize=9)
    ax[1, j].imshow(m, cmap=cmap, vmin=-0.5, vmax=N_CLASSES - 0.5, interpolation='nearest')
for a in ax.ravel():
    a.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'examples.png', dpi=100); plt.show()

frames.drop(columns=['path']).to_csv(OUT / 'frames.csv', index=False)
shutil.make_archive(str(WORK / 'results_01'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))